## Setup

In [1]:
# Setup — run this cell first
import numpy as np                # numpy = fast maths on lists of numbers
import pandas as pd               # pandas = tool for working with tables of data
import matplotlib.pyplot as plt   # matplotlib = tool for drawing charts
import seaborn as sns             # seaborn = prettier chart styling
import warnings                   # lets us hide harmless warning messages

from sklearn.model_selection import train_test_split   # splits data into train vs test
from sklearn.compose import ColumnTransformer           # applies different prep to different columns
from sklearn.pipeline import Pipeline                   # chains prep steps + model into one object
from sklearn.preprocessing import StandardScaler, OneHotEncoder   # scaling numbers / turning categories into 0-1 columns
from sklearn.impute import SimpleImputer                # fills in missing values

warnings.filterwarnings("ignore")   # hide harmless warnings so output stays clean
pd.set_option("display.max_columns", 20)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (11, 4.5)

print("✅ Libraries loaded — you're ready to go!")

✅ Libraries loaded — you're ready to go!


## Step 1 — Load the data and define X, y

In [2]:
df = pd.read_csv("data/credit_card_fraud_2026.csv")

# Separate features (X) from target (y)
# y = the answer we want to predict; X = the clues the model can use
y = df["is_fraud"]
X = df.drop(columns=["transaction_id", "is_fraud"])

print(f"X shape: {X.shape}  (20,000 customers × 10 features)")
print(f"y shape: {y.shape}  (20,000 labels — 1.7% fraud cases)")
print()
print("Feature columns:")
for col in X.columns:
    dtype = X[col].dtype
    print(f"  · {col:25s}  {dtype}")

X shape: (20000, 24)  (20,000 customers × 10 features)
y shape: (20000,)  (20,000 labels — 1.7% fraud cases)

Feature columns:
  · amount_usd                 float64
  · merchant_category          str
  · card_type                  str
  · auth_method                str
  · channel                    str
  · device_type                str
  · is_foreign_transaction     bool
  · hours_since_last_txn       float64
  · txn_count_last_24h         int64
  · distance_from_home_km      float64
  · card_age_months            int64
  · customer_age               int64
  · account_balance_usd        float64
  · is_new_merchant            bool
  · used_vpn                   bool
  · ip_country_mismatch        bool
  · billing_shipping_mismatch  bool
  · cvv_retry_count            int64
  · velocity_score             float64
  · time_of_day_hour           int64
  · day_of_week                int64
  · is_ai_generated_scam_attempt  bool
  · merchant_risk_score        float64
  · prior_disputes     

## Step 2 — Split BEFORE preprocessing (the leakage rule)

In [3]:
# Hold back 20% of customers as a test set the model never sees during training.
# stratify=y keeps the churn percentage identical in both halves.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

print(f"Training set: {len(X_train):,} customers ({y_train.mean():.1%} fraud rate)")
print(f"Test set:     {len(X_test):,} customers ({y_test.mean():.1%} fraud rate)")
print()
print("Class balance is preserved in both splits because of stratify=y.")

Training set: 16,000 customers (1.7% fraud rate)
Test set:     4,000 customers (1.7% fraud rate)

Class balance is preserved in both splits because of stratify=y.


## Step 3 — Identify numerical vs categorical features

In [8]:
numeric_features = [
    "amount_usd",
    "hours_since_last_txn",
    "txn_count_last_24h",
    "distance_from_home_km",
    "card_age_months",
    "customer_age",
    "account_balance_usd",
    "cvv_retry_count",
    "velocity_score",
    "time_of_day_hour",
    "day_of_week",
    "merchant_risk_score",
    "prior_disputes",
]
categorical_features = ["merchant_category", "card_type", "auth_method", "channel", "device_type", "is_foreign_transaction", "is_new_merchant", "used_vpn", "ip_country_mismatch", "billing_shipping_mismatch", "is_ai_generated_scam_attempt"]

print("Numerical features (impute → scale):")
for c in numeric_features: print(f"  · {c}")
print()
print("Categorical features (impute → one-hot encode):")
for c in categorical_features: print(f"  · {c}")

Numerical features (impute → scale):
  · amount_usd
  · hours_since_last_txn
  · txn_count_last_24h
  · distance_from_home_km
  · card_age_months
  · customer_age
  · account_balance_usd
  · cvv_retry_count
  · velocity_score
  · time_of_day_hour
  · day_of_week
  · merchant_risk_score
  · prior_disputes

Categorical features (impute → one-hot encode):
  · merchant_category
  · card_type
  · auth_method
  · channel
  · device_type
  · is_foreign_transaction
  · is_new_merchant
  · used_vpn
  · ip_country_mismatch
  · billing_shipping_mismatch
  · is_ai_generated_scam_attempt


## Step 4 — Build the ColumnTransformer

In [10]:
# Mini-pipeline for numerical columns
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

# Mini-pipeline for categorical columns
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# Combine into a single ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

print("✅ Preprocessor built.")
print()
print("It will apply, in order:")
print("  Numerical columns:")
print("    1. SimpleImputer(strategy='median')   → fill NaN with column median")
print("    2. StandardScaler()                    → subtract mean, divide by std")
print()
print("  Categorical columns:")
print("    1. SimpleImputer(strategy='most_frequent')  → fill NaN with most common value")
print("    2. OneHotEncoder(handle_unknown='ignore')   → expand to 0/1 columns")

✅ Preprocessor built.

It will apply, in order:
  Numerical columns:
    1. SimpleImputer(strategy='median')   → fill NaN with column median
    2. StandardScaler()                    → subtract mean, divide by std

  Categorical columns:
    1. SimpleImputer(strategy='most_frequent')  → fill NaN with most common value
    2. OneHotEncoder(handle_unknown='ignore')   → expand to 0/1 columns


## Step 5 — Fit and transform the training data

In [11]:
# fit_transform = LEARN the medians/means from training data, then APPLY them
X_train_processed = preprocessor.fit_transform(X_train)

# Get the resulting column names (numeric names unchanged + one-hot columns named region_X, subscription_tier_X)
feature_names_out = preprocessor.get_feature_names_out()

print(f"Original X_train shape:    {X_train.shape}  ({X_train.shape[1]} columns)")
print(f"Processed X_train shape:   {X_train_processed.shape}  ({X_train_processed.shape[1]} columns)")
print()
print(f"Number of NaN values in processed array: {np.isnan(X_train_processed).sum()}")
print()
print("Final feature names:")
for name in feature_names_out:
    print(f"  · {name}")

Original X_train shape:    (16000, 24)  (24 columns)
Processed X_train shape:   (16000, 60)  (60 columns)

Number of NaN values in processed array: 0

Final feature names:
  · num__amount_usd
  · num__hours_since_last_txn
  · num__txn_count_last_24h
  · num__distance_from_home_km
  · num__card_age_months
  · num__customer_age
  · num__account_balance_usd
  · num__cvv_retry_count
  · num__velocity_score
  · num__time_of_day_hour
  · num__day_of_week
  · num__merchant_risk_score
  · num__prior_disputes
  · cat__merchant_category_Crypto Exchange
  · cat__merchant_category_Electronics
  · cat__merchant_category_Fuel
  · cat__merchant_category_Gaming
  · cat__merchant_category_Gift Cards
  · cat__merchant_category_Groceries
  · cat__merchant_category_Healthcare
  · cat__merchant_category_Online Retail
  · cat__merchant_category_Restaurants
  · cat__merchant_category_Streaming
  · cat__merchant_category_Travel
  · cat__merchant_category_Utilities
  · cat__card_type_Amex
  · cat__card_type_Di

## Check the data if standard scaling is applied

In [15]:
# Show the effect of scaling for one column — account_balance_usd
# Find where the spend column ended up in the processed array
col_idx = list(feature_names_out).index("num__account_balance_usd")

before = X_train["account_balance_usd"].dropna()
after  = X_train_processed[:, col_idx]

print(f"Before scaling — num__account_balance_usd:")
print(f"  mean: {before.mean():7.2f}     std: {before.std():7.2f}")
print(f"  min:  {before.min():7.2f}     max: {before.max():7.2f}")
print()
print(f"After scaling — num__account_balance_usd:")
print(f"  mean: {after.mean():7.2f}     std: {after.std():7.2f}")
print(f"  min:  {after.min():7.2f}     max: {after.max():7.2f}")
print()
print("→ Centred on 0, std ≈ 1. The original distribution shape is preserved, only the scale changed.")

Before scaling — num__account_balance_usd:
  mean: 3319.29     std: 4398.85
  min:    54.11     max: 127125.86

After scaling — num__account_balance_usd:
  mean:   -0.00     std:    1.00
  min:    -0.74     max:   28.15

→ Centred on 0, std ≈ 1. The original distribution shape is preserved, only the scale changed.


## ✅ Section Summary

| Step | What it does | sklearn class |
|---|---|---|
| **1. Split first** | Hold out 20% as a test set BEFORE fitting any preprocessing | `train_test_split(stratify=y)` |
| **2. Median impute** | Fill numerical NaN with the column median (robust to outliers) | `SimpleImputer(strategy="median")` |
| **3. Standard scale** | Centre each numeric column on 0, std 1 | `StandardScaler()` |
| **4. Mode impute** | Fill categorical NaN with the most common value | `SimpleImputer(strategy="most_frequent")` |
| **5. One-hot encode** | Expand each categorical to binary columns | `OneHotEncoder(handle_unknown="ignore")` |
| **6. ColumnTransformer** | Apply different pipelines to different column sets | `ColumnTransformer([("num", ...), ("cat", ...)])` |
---
**Up next → Part 2:** Train the logistic regression on the preprocessed data, inspects the coefficients, and uses k-fold cross-validation to get a stable accuracy estimate.
Open `03_train_validate.ipynb`